# 18.2 Authentication and Handling Secrets

**Prerequisites:** 18.1 REST and JSON, 17.2 pyproject.toml, 15.10 Logging  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- The four schemes you will actually meet: API key, **Bearer**, Basic, HMAC
- 🔴 `401` vs `403` — and why only one of them is worth retrying
- Short-lived tokens, expiry, and refresh
- 🔴 **Never hardcode a secret** — environment variables, `.env`, secret managers
- 🔴 Why a key in the **URL** is worse than the same key in a header
- Redacting credentials from logs and tracebacks (**15.10**)
- A real case study: the credentials in **this repository's own history**
- What to do when a secret leaks — and what does *not* help

---

## Proving who you are

Almost every useful API needs to know who is calling. Four schemes cover nearly everything:

| Scheme | Looks like | Used by |
|---|---|---|
| **API key** | `X-API-Key: abc123` header | simple services |
| 🔴 **Bearer token** | `Authorization: Bearer eyJ...` | **most modern APIs**, OAuth2, JWT |
| **Basic** | `Authorization: Basic base64(user:pass)` | legacy, internal tools |
| **HMAC signature** | `X-Signature: <hmac of the request>` | webhooks, payment APIs |

They differ in *what* travels and *how often*. What they share is that **something secret leaves
your machine**, which is what the second half of this notebook is about.

> The fake API below enforces all four for real — wrong credentials genuinely get `401`.

In [ ]:
# ---- A fake API that actually checks credentials ----
import base64
import hashlib
import hmac
import http.server
import json
import socket
import threading
import time
import urllib.parse

VALID_TOKEN = "tok_live_4f9a2b7c"          # pretend this is real
VALID_BASIC = ("aditya", "correct-horse")
SIGNING_KEY = b"shared-signing-secret"
ISSUED_TOKENS = {}                          # token -> expiry timestamp

REQUEST_LOG = []


class SecureAPI(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        """Silence the default logging."""

    def _send(self, status, payload=None, headers=None):
        extra = dict(headers or {})
        if payload is None:
            self.send_response(status)
            extra["Content-Length"] = "0"
            for key, value in extra.items():
                self.send_header(key, value)
            self.end_headers()
            return
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for key, value in extra.items():
            self.send_header(key, value)
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        parsed = urllib.parse.urlparse(self.path)
        query = urllib.parse.parse_qs(parsed.query)
        auth = self.headers.get("Authorization", "")
        REQUEST_LOG.append((self.path, auth))

        # 1. Bearer token
        if parsed.path == "/v1/jobs":
            if not auth.startswith("Bearer "):
                return self._send(401, {"error": "missing bearer token"},
                                  {"WWW-Authenticate": 'Bearer realm="jobs"'})
            token = auth.removeprefix("Bearer ")
            if token in ISSUED_TOKENS:
                if time.time() > ISSUED_TOKENS[token]:
                    return self._send(401, {"error": "token expired"})
            elif token != VALID_TOKEN:
                return self._send(401, {"error": "invalid token"})
            return self._send(200, {"items": [{"id": "build-001", "state": "done"}]})

        # 2. Scope enforcement - authenticated but not permitted
        if parsed.path == "/v1/admin":
            if auth != f"Bearer {VALID_TOKEN}":
                return self._send(401, {"error": "invalid token"})
            return self._send(403, {"error": "insufficient scope",
                                    "required": "admin:write"})

        # 3. HTTP Basic
        if parsed.path == "/v1/basic":
            if not auth.startswith("Basic "):
                return self._send(401, {"error": "basic auth required"},
                                  {"WWW-Authenticate": 'Basic realm="jobs"'})
            decoded = base64.b64decode(auth.removeprefix("Basic ")).decode()
            user, _, password = decoded.partition(":")
            if (user, password) != VALID_BASIC:
                return self._send(401, {"error": "bad credentials"})
            return self._send(200, {"user": user})

        # 4. API key in the QUERY STRING - the bad pattern, kept to demonstrate why
        if parsed.path == "/v1/legacy":
            key = query.get("api_key", [""])[0]
            if key != VALID_TOKEN:
                return self._send(401, {"error": "bad api_key"})
            return self._send(200, {"warning": "key travelled in the URL"})

        # 5. HMAC-signed request
        if parsed.path == "/v1/signed":
            signature = self.headers.get("X-Signature", "")
            timestamp = self.headers.get("X-Timestamp", "")
            expected = hmac.new(SIGNING_KEY,
                                f"GET/v1/signed{timestamp}".encode(),
                                hashlib.sha256).hexdigest()
            if not hmac.compare_digest(signature, expected):
                return self._send(401, {"error": "bad signature"})
            return self._send(200, {"verified": True})

        return self._send(404, {"error": "not found"})

    def do_POST(self):
        parsed = urllib.parse.urlparse(self.path)
        length = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(length) or b"{}")
        REQUEST_LOG.append((self.path, self.headers.get("Authorization", "")))

        # A token endpoint: exchange a long-lived secret for a short-lived token
        if parsed.path == "/oauth/token":
            if payload.get("client_secret") != "cs_9f3e":
                return self._send(401, {"error": "invalid_client"})
            token = f"tok_short_{int(time.time() * 1000) % 100000}"
            ISSUED_TOKENS[token] = time.time() + payload.get("ttl", 2)
            return self._send(200, {"access_token": token,
                                    "token_type": "Bearer",
                                    "expires_in": payload.get("ttl", 2)})
        return self._send(404, {"error": "not found"})


class QuietServer(http.server.ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, *args):
        """A client hanging up is normal."""


def start_api():
    probe = socket.socket()
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
    probe.close()
    server = QuietServer(("127.0.0.1", port), SecureAPI)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{port}"


SERVER, BASE = start_api()
print("secured fake API on", BASE)

## Bearer tokens, and the `401`/`403` distinction

In [ ]:
import requests

session = requests.Session()

print("--- no credentials at all ---")
anonymous = session.get(f"{BASE}/v1/jobs", timeout=5)
print(f"   {anonymous.status_code} {anonymous.json()}")
print(f"   WWW-Authenticate: {anonymous.headers.get('WWW-Authenticate')}")

print()
print("--- a wrong token ---")
wrong = session.get(f"{BASE}/v1/jobs", timeout=5,
                    headers={"Authorization": "Bearer tok_not_real"})
print(f"   {wrong.status_code} {wrong.json()}")

print()
print("--- the right token ---")
good = session.get(f"{BASE}/v1/jobs", timeout=5,
                   headers={"Authorization": f"Bearer {VALID_TOKEN}"})
print(f"   {good.status_code} {good.json()}")

print()
print("--- the RIGHT token, on an endpoint it may not touch ---")
forbidden = session.get(f"{BASE}/v1/admin", timeout=5,
                        headers={"Authorization": f"Bearer {VALID_TOKEN}"})
print(f"   {forbidden.status_code} {forbidden.json()}")

🔴 **Read the last two together.** Same token, two outcomes:

| | Means | Retry? |
|---|---|---|
| `401 Unauthorized` | *"I do not know who you are"* — missing, malformed or **expired** credentials | ✅ **after refreshing the token** |
| `403 Forbidden` | *"I know who you are, and no"* — the credential is valid but lacks the scope | ❌ **never** — retrying cannot help |

Getting this wrong produces one of two bugs: a client that hammers a `403` forever, or one that
gives up on an expired token it could simply have refreshed.

Note the **`WWW-Authenticate`** header on the `401` — the server telling you which scheme it
expects. Almost nobody reads it; it is the fastest way to find out what an API wants.

## Setting credentials once

Putting the header on every call is repetitive and easy to forget. Set it on the `Session`, or
use an auth object.

In [ ]:
from requests.auth import AuthBase, HTTPBasicAuth


class BearerAuth(AuthBase):
    """A reusable auth object - requests calls it for every request."""

    def __init__(self, token):
        self._token = token

    def __call__(self, request):
        request.headers["Authorization"] = f"Bearer {self._token}"
        return request

    def __repr__(self):
        return "BearerAuth(token=***)"          # 🔴 never repr the secret


# 1. On the session - applies to everything
client = requests.Session()
client.auth = BearerAuth(VALID_TOKEN)
print("session auth :", client.get(f"{BASE}/v1/jobs", timeout=5).status_code)

# 2. Per request, when you need a different identity
print("per request  :", session.get(f"{BASE}/v1/jobs", timeout=5,
                                    auth=BearerAuth(VALID_TOKEN)).status_code)

# 3. Basic auth is built in
basic = session.get(f"{BASE}/v1/basic", timeout=5,
                    auth=HTTPBasicAuth(*VALID_BASIC))
print("basic auth   :", basic.status_code, basic.json())

print()
print("what Basic actually sends:")
print("   ", REQUEST_LOG[-1][1])
print("   🔴 base64 is ENCODING, not encryption - trivially reversible.")
import base64 as b64
print("   decoded:", b64.b64decode(REQUEST_LOG[-1][1].split()[1]).decode())
print("   Basic auth is only safe over HTTPS. So is everything else here.")

🔴 **`base64` is not encryption.** HTTP Basic sends your password in a form
anyone can reverse in one line — which is why it is acceptable *only* over HTTPS, and why
`__repr__` above deliberately prints `***`.

## Short-lived tokens and refresh

A long-lived secret that leaks is a long-lived problem. The standard fix is to exchange it,
rarely, for a **short-lived access token** — so a leaked token expires on its own.

```
   client_secret  ──POST /oauth/token──>  access_token (expires in 300s)
   (stored once,                          (used for every call,
    rotated rarely)                        refreshed when it expires)
```

In [ ]:
import time


class RefreshingClient:
    """Fetches a short-lived token and renews it when the API says 401."""

    def __init__(self, base, client_secret, ttl=1.0):
        self._base = base
        self._secret = client_secret
        self._ttl = ttl
        self._session = requests.Session()
        self._token = None
        self.refreshes = 0

    def _fetch_token(self):
        response = self._session.post(f"{self._base}/oauth/token", timeout=5,
                                      json={"client_secret": self._secret,
                                            "ttl": self._ttl})
        response.raise_for_status()
        self.refreshes += 1
        self._token = response.json()["access_token"]

    def get(self, path):
        if self._token is None:
            self._fetch_token()
        response = self._session.get(f"{self._base}{path}", timeout=5,
                                     headers={"Authorization": f"Bearer {self._token}"})
        if response.status_code == 401:              # 🔴 401 only - never on 403
            self._fetch_token()
            response = self._session.get(
                f"{self._base}{path}", timeout=5,
                headers={"Authorization": f"Bearer {self._token}"})
        response.raise_for_status()
        return response.json()


api = RefreshingClient(BASE, client_secret="cs_9f3e", ttl=1.0)

print("first call  :", api.get("/v1/jobs"), f"(refreshes: {api.refreshes})")
time.sleep(1.2)                                       # let the token expire
print("after expiry:", api.get("/v1/jobs"), f"(refreshes: {api.refreshes})")
print()
print("🔴 The second call got a 401, refreshed transparently, and retried once.")
print("   Retry ONCE - a refresh loop against a genuinely bad secret is a")
print("   denial-of-service attack on your own credentials.")

bad = RefreshingClient(BASE, client_secret="wrong-secret")
try:
    bad.get("/v1/jobs")
except requests.HTTPError as exc:
    print(f"\n   a bad client_secret fails fast: {exc.response.status_code}"
          f" {exc.response.json()}")

## 🔴 Where secrets must not go

This is the part that matters most, and the failure modes are all mundane.

| Never | Why |
|---|---|
| **Hardcoded in source** | it reaches version control, and history is forever |
| **In a URL / query string** | logged by every proxy, browser, and the server's access log |
| **In a log line** | logs get shipped, indexed and shared (**15.10**) |
| **In a traceback or error report** | crash reporters send them off-machine (**15.7**) |
| **In a notebook output cell** | 🔴 the output is *saved in the file* |
| **In a git commit — even if deleted later** | it stays in the history |

### The URL problem, demonstrated

An API key in a header and the same key in a query string are equally secret in transit — both
are inside the TLS tunnel. The difference is **everything that writes down the URL**.

In [ ]:
legacy = session.get(f"{BASE}/v1/legacy",
                     params={"api_key": VALID_TOKEN, "region": "eu"}, timeout=5)

print("the request succeeded:", legacy.status_code, legacy.json())
print()
print("but look what is now in the URL object:")
print("   ", legacy.url)
print()
print("...and in the server's own request log:")
print("   ", REQUEST_LOG[-1][0])
print()
print("🔴 That URL is written to: the server access log, every proxy in between,")
print("   your own logs if you log response.url, browser history, and any")
print("   error tracker that captures the request. The header version is not.")

print()
header_version = session.get(f"{BASE}/v1/jobs", timeout=5,
                             headers={"Authorization": f"Bearer {VALID_TOKEN}"})
print("with a header instead:")
print("    url logged :", header_version.url)
print("    secret in it:", VALID_TOKEN in header_version.url)

The URL still holds the key in the first case and does not in the second.
**Use a header.** If an API only supports a key in the query string, treat that as a reason to
rotate it more often.

## Loading secrets properly

The baseline is an **environment variable**: not in your source, not in your repository, set
differently in development and production.

```python
import os

token = os.environ["JOBS_API_TOKEN"]        # 🔴 KeyError if missing - good
token = os.getenv("JOBS_API_TOKEN")         # None if missing - fails later, worse
```

🔴 **Prefer `os.environ[...]`.** Failing at startup with a clear `KeyError` beats a `None`
travelling into a request and coming back as a confusing `401`.

In [ ]:
import os
import textwrap
import tempfile
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py182_"))

# A .env file - never committed; listed in .gitignore
env_file = WORK / ".env"
env_file.write_text(textwrap.dedent("""
    # local development only
    JOBS_API_TOKEN=tok_live_4f9a2b7c
    JOBS_API_BASE=http://127.0.0.1:9999
    # a comment, and a blank line follow

""").lstrip(), encoding="utf-8")


def load_dotenv(path):
    """A 12-line .env loader. `python-dotenv` does this with more features."""
    values = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip("\"'")
    return values


loaded = load_dotenv(env_file)
print("parsed from .env:", {k: "***" for k in loaded})     # 🔴 never print values

# Environment first, .env as a fallback: real deployments set real env vars.
os.environ.setdefault("JOBS_API_TOKEN", loaded["JOBS_API_TOKEN"])

token = os.environ["JOBS_API_TOKEN"]
print("token loaded   :", f"{token[:8]}... ({len(token)} chars)")

print()
print("--- what happens when it is missing ---")
for name in ("JOBS_API_TOKEN", "JOBS_API_MISSING"):
    try:
        os.environ[name]
        print(f"   os.environ[{name!r}]  -> found")
    except KeyError:
        print(f"   os.environ[{name!r}] -> 🔴 KeyError, at startup, where you want it")
    print(f"   os.getenv({name!r})   -> {os.getenv(name) and 'found' or None}"
          f"{'   <- None travels onward silently' if not os.getenv(name) else ''}")

> **`python-dotenv`** does the above properly (quoting, interpolation,
> `override=`). The point of writing it out is that a `.env` file is not magic — it is a text
> file you must add to `.gitignore` and never commit.

**Beyond `.env`**, in rough order of seriousness:

| Approach | Where the secret lives |
|---|---|
| Environment variable | the process environment |
| `.env` file (gitignored) | disk, development only |
| **OS keyring** (`keyring`) | the platform credential store |
| **Secret manager** | AWS Secrets Manager, Vault, GCP Secret Manager — with audit and rotation |
| 🔴 **Workload identity** | no secret exists at all — the runtime proves who it is |

That last row is the same idea as trusted publishing in **17.3**: the safest credential is one
that does not exist.

## 🔴 Keeping secrets out of logs

A token in a log file is a leaked token. **15.10** built a logging system; this is the filter it
needs.

In [ ]:
import logging
import re
import sys


class RedactingFormatter(logging.Formatter):
    """Mask anything that looks like a credential before it is written."""

    PATTERNS = [
        re.compile(r"(Bearer\s+)[A-Za-z0-9._-]+"),
        re.compile(r"((?:api_key|token|password|secret)=)[^&\s\"']+", re.IGNORECASE),
        re.compile(r"(tok_[a-z]+_)[A-Za-z0-9]+"),
    ]

    def format(self, record):
        message = super().format(record)
        for pattern in self.PATTERNS:
            message = pattern.sub(r"\1***REDACTED***", message)
        return message


stream = logging.StreamHandler(sys.stdout)
stream.setFormatter(RedactingFormatter("%(levelname)-8s %(message)s"))
log = logging.getLogger("api-client")
log.handlers.clear()
log.addHandler(stream)
log.setLevel(logging.INFO)
log.propagate = False

print("--- what a careless client logs ---")
log.info("GET %s", f"{BASE}/v1/legacy?api_key={VALID_TOKEN}&region=eu")
log.info("sending header Authorization: Bearer %s", VALID_TOKEN)
log.warning("auth failed for token=%s", VALID_TOKEN)

print()
print("🔴 Redaction at the formatter is a SAFETY NET, not a strategy.")
print("   It only catches patterns you thought of. The real fix is never")
print("   putting the secret in the message - log the request ID instead.")

Every credential was masked on the way out — but read the closing note. A
redacting formatter catches **the patterns you predicted**. It is a second line of defence
behind *not logging the secret in the first place*.

## 🔴 A real case study: this repository

This is not hypothetical. **This curriculum's own git history contains real credentials.**

- Commit `b0537f6` — the original 2019 notes — included a **Gmail address and password** passed
  straight to `server.login()`, and a 32-character **OpenWeatherMap API key**, twice.
- Commit `27ef473` removed them from the working tree.
- 🔴 **They remained fully readable in `b0537f6`.** Deleting a file, and even deleting the
  entire folder later, changed nothing about the history.

What actually resolved it: the author **changed the password and deleted the API key**. Rotation
— not deletion, not history rewriting.

### What to do when a secret leaks

| Step | Detail |
|---|---|
| **1. Rotate it. Immediately.** | 🔴 This is the only step that truly matters. Assume it is compromised. |
| 2. Revoke the old one | not just replace — the old value must stop working |
| 3. Check the audit log | was it used by anyone else, and when? |
| 4. Then clean up | `git filter-repo`, or leave it; the value is inert now |
| 5. Add a scanner | `gitleaks`, `detect-secrets`, or GitHub push protection |

**What does *not* help:**

- ❌ Deleting the file — history keeps it
- ❌ Force-pushing — clones, forks and CI caches keep it
- ❌ Making the repository private — it may already be indexed
- ❌ Hoping nobody noticed — bots scan public commits within **seconds** of a push

> A pre-commit hook (**17.4**) running a secret scanner is the cheapest possible insurance. The
> incident above happened in 2019 and was only found by an audit six years later.

In [ ]:
# ---- tidy up ----
SERVER.shutdown()
import shutil
shutil.rmtree(WORK, ignore_errors=True)
print("fake API stopped, scratch removed:", not WORK.exists())
print(f"requests handled: {len(REQUEST_LOG)}")
print()
print("🔴 Note what this notebook never did: it never printed a real secret")
print("   into an output cell. Notebook outputs are saved INTO the file.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Hardcoding a credential in source.** It reaches version control, and history is permanent — as this repository's own `b0537f6` demonstrates.
2. 🔴 **Putting a key in the query string.** Every proxy, access log and error tracker writes the URL down. Use a header.
3. **Treating `403` like `401`.** Refreshing a token cannot fix a permissions problem.
4. **Retrying a refresh in a loop.** Against a genuinely bad secret you have built a denial-of-service attack on your own account.
5. **Using `os.getenv()` for a required secret.** `None` travels onward and surfaces as a confusing `401`; `os.environ[...]` fails at startup.
6. **Committing `.env`.** Add it to `.gitignore` before you create it, not after.
7. 🔴 **Printing a secret into a notebook output cell.** The output is saved in the file and committed with it.
8. **Assuming `base64` protects anything.** HTTP Basic is reversible in one line; only TLS protects it.
9. **Relying on log redaction as the strategy.** It only catches patterns you predicted.
10. **Deleting a leaked secret instead of rotating it.** Deletion changes nothing; rotation is the fix.

## Best Practices

- Read secrets from the environment; fail loudly at startup when one is missing.
- Send credentials in headers, never in URLs.
- Prefer short-lived tokens exchanged from a long-lived secret, and refresh on `401` only.
- Scope credentials as narrowly as the API allows — one project, one permission.
- Set auth once on a `Session` or an `AuthBase` object rather than per call.
- Give any object holding a secret a `__repr__` that masks it.
- Add a redacting formatter (**15.10**) as a safety net, and do not log secrets anyway.
- Run a secret scanner in a pre-commit hook (**17.4**) and in CI.
- Rotate on any suspicion. It is cheap; a leak is not.
- Prefer workload identity where the platform offers it — no secret is the safest secret.

## Practice Exercises

Try these before moving on.

1. Call `/v1/jobs` with no token, a bad token and a good one. Then call `/v1/admin` with the good token. Explain each status code in one sentence.
2. Write an `AuthBase` subclass for an `X-API-Key` header, with a `__repr__` that masks the key. Prove the mask works.
3. 🔴 Extend `RefreshingClient` so it refuses to refresh more than once per call. What happens today if the secret is revoked mid-session?
4. Send a key as a query parameter and log `response.url`. Now find every place that string could end up. How many did you think of?
5. Write a `.env` loader that supports quoted values and `export ` prefixes. Compare yours with `python-dotenv`.
6. Add a pattern to `RedactingFormatter` for AWS keys (`AKIA...`). Then find one it still misses — that is the point of the closing note.
7. 🔴 Run `git log -p` on a repository of your own and search for `password`, `token`, `secret` and `api_key`. Found anything?
8. Set up `gitleaks` or `detect-secrets` as a pre-commit hook (**17.4**) and try to commit a fake key.
9. **Interview question:** you discover an API key in a public repository's history. Walk through the next hour, in order.

---

## Version notes

| Version | Change |
|---|---|
| **3.9** | `str.removeprefix()` — used above to strip `"Bearer "` cleanly |
| **3.6+** | `hmac.compare_digest()` — 🔴 **constant-time** comparison; `==` on a signature leaks timing information |
| **requests 2.x** | `AuthBase` is the extension point; `HTTPBasicAuth` and `HTTPDigestAuth` are built in |

> **On OAuth2.** The `/oauth/token` exchange above is the *client credentials* flow, the simplest
> of several. Authorisation-code flows with a browser redirect, PKCE and refresh tokens are a
> larger topic; the libraries to reach for are `authlib` or `requests-oauthlib` rather than
> hand-rolling.

## Where next

| Notebook | Covers |
|---|---|
| **18.3** | pagination, rate limits, retries and backoff |
| **18.4** | validating what you receive, with `pydantic` |
| **18.5** | testing API clients, and concurrency for I/O-bound work |

## Related

- **18.1** — status codes, `Session`, and the exception taxonomy
- **15.10 Logging** — the formatter the redaction filter plugs into
- **17.3 Packaging and Publishing** — trusted publishing: the same "no secret at all" idea
- **17.4 ruff** — where a secret-scanning pre-commit hook belongs
- **2.5 Dictionaries** — the note that pointed here for security